In [ ]:
"""
==================================================
ML LEARNING JOURNEY - DAY 101
==================================================
Week: 15 of 24
Day: 101 of 168
Date: May 17, 2026 
Topic: ROS2 Services & Actions

Week 15 Progress:
✅ Day 99:  ROS2 Setup & TurtleSim
✅ Day 100: ROS2 PubSub & Python Nodes
🔄 Day 101: ROS2 Services & Actions (TODAY!)
⬜ Day 102: ROS2 Packages
⬜ Day 103: Sensor Data Processing
⬜ Day 104: TurtleSim Controller — Build Day
⬜ Day 105: TurtleSim Controller — Polish & Demo

Progress: 28.6% (2/7 days)

==================================================
🎯 Week 15 Project: ROS2 & Robotics
- Install and configure ROS2 Humble
- Master nodes, topics, services, and actions
- Build a complete TurtleSim Controller package
- Process sensor data (camera, LiDAR)
- Create demo video and professional README

🎯 Today's Learning Objectives:
1. Understand Services — request/response communication
2. Create a service server in Python
3. Create a service client in Python
4. Understand Actions — goal/feedback/result communication
5. Compare Topics vs Services vs Actions — when to use each

📚 Today's Structure:
   Part 1 (2h): ROS2 Services
   Part 2 (2h): ROS2 Actions
   Part 3 (1h): Comparison & Practice

🎯 SUCCESS CRITERIA:
   ✅ Understand service request/response pattern
   ✅ Write a service server that handles turtle commands
   ✅ Write a service client that calls the server
   ✅ Understand action goal/feedback/result pattern
   ✅ Know exactly when to use Topics vs Services vs Actions

==================================================
"""

In [1]:
print("=" * 80)
print("🤖 ROS2 SERVICES & ACTIONS — Day 101")
print("=" * 80)
print("""
Today we complete the ROS2 communication trinity:
  ✅ Topics    (Day 100) — continuous data streams
  🔄 Services  (Today)  — request/response
  🔄 Actions   (Today)  — goal/feedback/result
""")
print("=" * 80)

🤖 ROS2 SERVICES & ACTIONS — Day 101

Today we complete the ROS2 communication trinity:
  ✅ Topics    (Day 100) — continuous data streams
  🔄 Services  (Today)  — request/response
  🔄 Actions   (Today)  — goal/feedback/result



In [2]:
print("\n" + "=" * 80)
print("🔧 PART 1: ROS2 SERVICES")
print("=" * 80)


🔧 PART 1: ROS2 SERVICES


In [3]:
# ==================================================
# EXERCISE 1.1: WHAT ARE SERVICES?
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.1: What are ROS2 Services?")
print("=" * 80)

"""
📖 THEORY: Services

A service is a synchronous request/response mechanism.
One node (the client) sends a request and WAITS for a response.
Another node (the server) receives the request, processes it,
and sends back a response.

Like an API call — you call it, you wait, you get a result.
"""

print("""
📋 SERVICE ARCHITECTURE:
─────────────────────────────────────────────────────────────

  Client Node                    Server Node
  ───────────                    ───────────
  sends Request  ────────────▶  receives Request
                                processes it
  receives Response ◀────────  sends Response

  Key properties:
  ├── Synchronous  — client blocks until response arrives
  ├── One-to-one   — one client, one server per call
  ├── Typed        — request and response have defined structure
  └── Reliable     — guaranteed delivery (unlike Best-Effort topics)

📋 SERVICE vs TOPIC:
─────────────────────────────────────────────────────────────

  Topic:                          Service:
  ─────────────────────────       ─────────────────────────────
  Continuous stream               One-time call
  Fire and forget                 Wait for response
  No acknowledgment               Guaranteed response
  Many-to-many                    One client ↔ one server
  Low latency                     Slightly higher latency

  Use Service when:
  ✅ You need a result back ("did it work?")
  ✅ The operation is quick (< 1 second)
  ✅ You want to trigger a specific action once
  ✅ Examples: spawn turtle, reset simulation, get robot state

  DON'T use Service when:
  ❌ The operation takes a long time (use Action instead)
  ❌ You need continuous data (use Topic instead)
  ❌ Many nodes need the result simultaneously (use Topic instead)

📋 TURTLESIM BUILT-IN SERVICES:
─────────────────────────────────────────────────────────────
  # List all available services:
  ros2 service list

  /turtlesim/reset                  → Reset canvas + turtle position
  /turtle1/teleport_absolute        → Jump turtle to exact coordinates
  /turtle1/teleport_relative        → Move turtle relative to current pose
  /turtle1/set_pen                  → Change pen color and width
  /spawn                            → Spawn a new turtle
  /kill                             → Remove a turtle

  # Inspect service type:
  ros2 service type /turtle1/teleport_absolute
  → turtlesim/srv/TeleportAbsolute

  # Show service interface (request + response):
  ros2 interface show turtlesim/srv/TeleportAbsolute
  →  float32 x
     float32 y
     float32 theta
     ---          ← separator between request and response
                  ← (empty response = just acknowledgment)
""")

print("\n✅ Exercise 1.1 Complete!")
print("=" * 80)


EXERCISE 1.1: What are ROS2 Services?

📋 SERVICE ARCHITECTURE:
─────────────────────────────────────────────────────────────

  Client Node                    Server Node
  ───────────                    ───────────
  sends Request  ────────────▶  receives Request
                                processes it
  receives Response ◀────────  sends Response

  Key properties:
  ├── Synchronous  — client blocks until response arrives
  ├── One-to-one   — one client, one server per call
  ├── Typed        — request and response have defined structure
  └── Reliable     — guaranteed delivery (unlike Best-Effort topics)

📋 SERVICE vs TOPIC:
─────────────────────────────────────────────────────────────

  Topic:                          Service:
  ─────────────────────────       ─────────────────────────────
  Continuous stream               One-time call
  Fire and forget                 Wait for response
  No acknowledgment               Guaranteed response
  Many-to-many                    

In [4]:
# ==================================================
# EXERCISE 1.2: SERVICE SERVER
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.2: Writing a Service Server")
print("=" * 80)

"""
📖 THEORY: Service Server

A service server:
1. Declares what service it provides (name + type)
2. Registers a callback that handles incoming requests
3. Returns a response for each request

We'll build a turtle commander service that accepts
named commands ("go_home", "spin", "stop") and executes them.
"""

turtle_commander_server_py = '''
# ============================================================
# turtle_commander_server.py — Service Server
# Provides a /turtle_command service that accepts named commands
#
# Save to: ~/ros2_ws/src/turtle_commander_server.py
# Run with: python3 turtle_commander_server.py
# ============================================================

import rclpy
from rclpy.node import Node
from std_srvs.srv import Trigger       # Simple service: no request data, bool response
from geometry_msgs.msg import Twist
from turtlesim.srv import TeleportAbsolute, SetPen


class TurtleCommanderServer(Node):
    """
    Service server that provides high-level turtle commands.
    Clients can call /turtle/go_home, /turtle/spin, /turtle/stop
    """

    def __init__(self):
        super().__init__("turtle_commander_server")

        # ── Service Servers ───────────────────────────────────
        # Each service takes: service_type, service_name, callback
        self.go_home_srv = self.create_service(
            Trigger,
            "/turtle/go_home",
            self.go_home_callback,
        )

        self.spin_srv = self.create_service(
            Trigger,
            "/turtle/spin",
            self.spin_callback,
        )

        self.stop_srv = self.create_service(
            Trigger,
            "/turtle/stop",
            self.stop_callback,
        )

        # ── Publisher for velocity commands ───────────────────
        self.cmd_publisher = self.create_publisher(
            Twist, "/turtle1/cmd_vel", 10
        )

        # ── Client to call TurtleSim's own services ───────────
        self.teleport_client = self.create_client(
            TeleportAbsolute, "/turtle1/teleport_absolute"
        )
        self.set_pen_client = self.create_client(
            SetPen, "/turtle1/set_pen"
        )

        # State
        self.is_spinning = False
        self.spin_timer = None

        self.get_logger().info("""
TurtleCommanderServer ready!
Services available:
  /turtle/go_home  → Teleport turtle to center
  /turtle/spin     → Start spinning in place
  /turtle/stop     → Stop all movement
""")

    # ── Service Callbacks ─────────────────────────────────────
    def go_home_callback(self, request, response):
        """
        Handle /turtle/go_home service call.
        Teleports turtle to center of canvas (5.5, 5.5).
        """
        self.get_logger().info("Service called: go_home")

        # Stop any current movement
        self._stop_movement()

        # Call TeleportAbsolute service
        teleport_req = TeleportAbsolute.Request()
        teleport_req.x = 5.544
        teleport_req.y = 5.544
        teleport_req.theta = 0.0

        # Wait for teleport service to be available
        if self.teleport_client.wait_for_service(timeout_sec=2.0):
            future = self.teleport_client.call_async(teleport_req)
            # Note: In production, use proper async handling
            # For simplicity here we use a brief spin
            rclpy.spin_until_future_complete(self, future, timeout_sec=2.0)
            response.success = True
            response.message = "Turtle teleported to home position (5.5, 5.5)"
        else:
            response.success = False
            response.message = "TeleportAbsolute service not available"

        self.get_logger().info(f"go_home result: {response.message}")
        return response

    def spin_callback(self, request, response):
        """Handle /turtle/spin — start spinning in place."""
        self.get_logger().info("Service called: spin")

        if self.is_spinning:
            response.success = False
            response.message = "Turtle is already spinning"
            return response

        # Start spin timer at 10 Hz
        self.is_spinning = True
        self.spin_timer = self.create_timer(0.1, self._spin_tick)

        response.success = True
        response.message = "Turtle is now spinning — call /turtle/stop to stop"
        return response

    def stop_callback(self, request, response):
        """Handle /turtle/stop — stop all movement."""
        self.get_logger().info("Service called: stop")

        self._stop_movement()

        response.success = True
        response.message = "Turtle stopped"
        return response

    # ── Helper Methods ────────────────────────────────────────
    def _spin_tick(self):
        """Timer callback: publish spin command."""
        msg = Twist()
        msg.linear.x = 0.0
        msg.angular.z = 2.0
        self.cmd_publisher.publish(msg)

    def _stop_movement(self):
        """Stop turtle and cancel spin timer if active."""
        if self.spin_timer:
            self.spin_timer.cancel()
            self.spin_timer = None
            self.is_spinning = False

        stop_msg = Twist()
        self.cmd_publisher.publish(stop_msg)


def main(args=None):
    rclpy.init(args=args)
    node = TurtleCommanderServer()
    try:
        rclpy.spin(node)
    except KeyboardInterrupt:
        pass
    finally:
        node.destroy_node()
        rclpy.shutdown()


if __name__ == "__main__":
    main()
'''

print("📄 turtle_commander_server.py:")
print(turtle_commander_server_py)

print("""
📋 KEY PARTS EXPLAINED:

  self.create_service(Trigger, "/turtle/go_home", self.go_home_callback)
    → Trigger = service type (request: empty, response: bool + string)
    → "/turtle/go_home" = service name clients will call
    → self.go_home_callback = called when a client sends a request

  def go_home_callback(self, request, response):
    → request contains the client's input data
    → response is populated and returned
    → MUST return response at the end

  Trigger service interface:
    Request:  (empty — no data needed)
    ---
    Response: bool success
              string message
""")

print("\n✅ Exercise 1.2 Complete!")
print("=" * 80)


EXERCISE 1.2: Writing a Service Server
📄 turtle_commander_server.py:

# ============================================================
# turtle_commander_server.py — Service Server
# Provides a /turtle_command service that accepts named commands
#
# Save to: ~/ros2_ws/src/turtle_commander_server.py
# Run with: python3 turtle_commander_server.py
# ============================================================

import rclpy
from rclpy.node import Node
from std_srvs.srv import Trigger       # Simple service: no request data, bool response
from geometry_msgs.msg import Twist
from turtlesim.srv import TeleportAbsolute, SetPen


class TurtleCommanderServer(Node):
    """
    Service server that provides high-level turtle commands.
    Clients can call /turtle/go_home, /turtle/spin, /turtle/stop
    """

    def __init__(self):
        super().__init__("turtle_commander_server")

        # ── Service Servers ───────────────────────────────────
        # Each service takes: service_type, service_

In [5]:
# ==================================================
# EXERCISE 1.3: SERVICE CLIENT
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 1.3: Writing a Service Client")
print("=" * 80)

turtle_commander_client_py = '''
# ============================================================
# turtle_commander_client.py — Service Client
# Calls the TurtleCommanderServer services
#
# Save to: ~/ros2_ws/src/turtle_commander_client.py
# Run with: python3 turtle_commander_client.py
# ============================================================

import rclpy
from rclpy.node import Node
from std_srvs.srv import Trigger
import sys
import time


class TurtleCommanderClient(Node):
    """
    Service client that calls TurtleCommanderServer.
    Usage: python3 turtle_commander_client.py [go_home|spin|stop]
    """

    def __init__(self):
        super().__init__("turtle_commander_client")

        # Create clients for each service
        self.go_home_client = self.create_client(Trigger, "/turtle/go_home")
        self.spin_client     = self.create_client(Trigger, "/turtle/spin")
        self.stop_client     = self.create_client(Trigger, "/turtle/stop")

    def wait_for_services(self, timeout_sec=5.0) -> bool:
        """Wait for all services to become available."""
        services = [
            ("/turtle/go_home", self.go_home_client),
            ("/turtle/spin",    self.spin_client),
            ("/turtle/stop",    self.stop_client),
        ]
        for name, client in services:
            if not client.wait_for_service(timeout_sec=timeout_sec):
                self.get_logger().error(
                    f"Service {name} not available after {timeout_sec}s"
                )
                return False
        return True

    def call_service(self, command: str) -> bool:
        """Call the requested command service and return success."""
        client_map = {
            "go_home": self.go_home_client,
            "spin":    self.spin_client,
            "stop":    self.stop_client,
        }

        if command not in client_map:
            self.get_logger().error(
                f"Unknown command: {command}. "
                f"Valid: {list(client_map.keys())}"
            )
            return False

        client = client_map[command]
        request = Trigger.Request()     # Empty request for Trigger

        self.get_logger().info(f"Calling service: /turtle/{command}")

        # Send async request
        future = client.call_async(request)

        # Wait for response (spin until done)
        rclpy.spin_until_future_complete(self, future, timeout_sec=5.0)

        if future.done():
            response = future.result()
            if response.success:
                self.get_logger().info(f"✅ Success: {response.message}")
            else:
                self.get_logger().warn(f"⚠️  Failed: {response.message}")
            return response.success
        else:
            self.get_logger().error("Service call timed out")
            return False


def main(args=None):
    rclpy.init(args=args)
    node = TurtleCommanderClient()

    # Get command from CLI argument
    command = sys.argv[1] if len(sys.argv) > 1 else "go_home"

    if not node.wait_for_services():
        node.get_logger().error("Services not available — is the server running?")
        rclpy.shutdown()
        return

    node.call_service(command)

    node.destroy_node()
    rclpy.shutdown()


if __name__ == "__main__":
    main()
'''

print("📄 turtle_commander_client.py:")
print(turtle_commander_client_py)

print("""
📋 HOW TO RUN:
─────────────────────────────────────────────────────────────
  Terminal 1: ros2 run turtlesim turtlesim_node
  Terminal 2: python3 turtle_commander_server.py
  Terminal 3: python3 turtle_commander_client.py go_home
              python3 turtle_commander_client.py spin
              python3 turtle_commander_client.py stop

  Or call directly from CLI (no client needed!):
  ros2 service call /turtle/go_home std_srvs/srv/Trigger
  ros2 service call /turtle/spin std_srvs/srv/Trigger
  ros2 service call /turtle/stop std_srvs/srv/Trigger

📋 KEY PARTS EXPLAINED:

  client.wait_for_service(timeout_sec=5.0)
    → Always wait before calling — server may not be ready yet
    → Returns True if available, False if timeout

  future = client.call_async(request)
    → Sends request without blocking
    → Returns a Future object

  rclpy.spin_until_future_complete(self, future, timeout_sec=5.0)
    → Processes callbacks until future is done
    → Essentially: "wait for the response"

  future.result()
    → Gets the response object once future is done
""")

print("\n✅ Exercise 1.3 Complete!")
print("=" * 80)


EXERCISE 1.3: Writing a Service Client
📄 turtle_commander_client.py:

# ============================================================
# turtle_commander_client.py — Service Client
# Calls the TurtleCommanderServer services
#
# Save to: ~/ros2_ws/src/turtle_commander_client.py
# Run with: python3 turtle_commander_client.py
# ============================================================

import rclpy
from rclpy.node import Node
from std_srvs.srv import Trigger
import sys
import time


class TurtleCommanderClient(Node):
    """
    Service client that calls TurtleCommanderServer.
    Usage: python3 turtle_commander_client.py [go_home|spin|stop]
    """

    def __init__(self):
        super().__init__("turtle_commander_client")

        # Create clients for each service
        self.go_home_client = self.create_client(Trigger, "/turtle/go_home")
        self.spin_client     = self.create_client(Trigger, "/turtle/spin")
        self.stop_client     = self.create_client(Trigger, "/turtle/sto

In [6]:
print("\n" + "=" * 80)
print("🎯 PART 2: ROS2 ACTIONS")
print("=" * 80)


🎯 PART 2: ROS2 ACTIONS


In [7]:
# ==================================================
# EXERCISE 2.1: WHAT ARE ACTIONS?
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.1: What are ROS2 Actions?")
print("=" * 80)

"""
📖 THEORY: Actions

Actions are for long-running tasks where you want:
- To send a goal (what to achieve)
- To receive feedback while it's in progress
- To get a final result when done
- To be able to cancel mid-way

Think of ordering food delivery:
- Goal: "Deliver a pizza to my address"
- Feedback: "Order confirmed", "Pizza being made", "Driver en route", "5 min away"
- Result: "Delivered successfully" or "Delivery failed"
- Cancel: "Actually, cancel my order"
"""

print("""
📋 ACTION ARCHITECTURE:
─────────────────────────────────────────────────────────────

  Client Node                         Server Node
  ───────────                         ───────────
  sends Goal ──────────────────────▶  accepts Goal
                                      executes task...
  receives Feedback ◀──────────────   publishes Feedback (periodic)
  receives Feedback ◀──────────────   publishes Feedback (periodic)
  receives Feedback ◀──────────────   publishes Feedback (periodic)
  receives Result ◀────────────────   sends final Result

  Optional: Client sends Cancel ───▶  Server stops task

📋 ACTION vs SERVICE vs TOPIC:
─────────────────────────────────────────────────────────────

  Feature           Topic      Service     Action
  ────────────────────────────────────────────────────────
  Communication     Stream     Req/Res     Goal/FB/Result
  Direction         Any        C → S       C → S
  Duration          Ongoing    Instant     Long-running
  Feedback          N/A        N/A         ✅ Yes
  Cancellable       N/A        No          ✅ Yes
  Use case          Sensors    Commands    Navigation

  Use Action when:
  ✅ Task takes > 1 second
  ✅ You want progress updates
  ✅ User might want to cancel
  ✅ Examples: navigate to point, pick up object, scan room

📋 ROS2 ACTION INTERFACE STRUCTURE:
─────────────────────────────────────────────────────────────
  # Example: nav2_msgs/action/NavigateToPose
  # Three sections separated by ---

  # Goal (what the client sends)
  geometry_msgs/PoseStamped pose
  string behavior_tree
  ---
  # Result (what the server sends when done)
  std_msgs/Empty result
  ---
  # Feedback (what the server sends periodically)
  geometry_msgs/PoseStamped current_pose
  float32 distance_remaining
  builtin_interfaces/Duration estimated_time_remaining

📋 CHECK TURTLESIM ACTIONS:
─────────────────────────────────────────────────────────────
  ros2 action list
  → /turtle1/rotate_absolute   ← TurtleSim has one built-in action!

  ros2 action info /turtle1/rotate_absolute
  ros2 interface show turtlesim/action/RotateAbsolute
  →
    # Goal
    float32 theta   ← target heading in radians
    ---
    # Result
    float32 delta   ← how much it actually rotated
    ---
    # Feedback
    float32 remaining  ← radians remaining to target

  # Call it from CLI:
  ros2 action send_goal /turtle1/rotate_absolute \\
    turtlesim/action/RotateAbsolute "{theta: 1.57}" \\
    --feedback
""")

print("\n✅ Exercise 2.1 Complete!")
print("=" * 80)


EXERCISE 2.1: What are ROS2 Actions?

📋 ACTION ARCHITECTURE:
─────────────────────────────────────────────────────────────

  Client Node                         Server Node
  ───────────                         ───────────
  sends Goal ──────────────────────▶  accepts Goal
                                      executes task...
  receives Feedback ◀──────────────   publishes Feedback (periodic)
  receives Feedback ◀──────────────   publishes Feedback (periodic)
  receives Feedback ◀──────────────   publishes Feedback (periodic)
  receives Result ◀────────────────   sends final Result

  Optional: Client sends Cancel ───▶  Server stops task

📋 ACTION vs SERVICE vs TOPIC:
─────────────────────────────────────────────────────────────

  Feature           Topic      Service     Action
  ────────────────────────────────────────────────────────
  Communication     Stream     Req/Res     Goal/FB/Result
  Direction         Any        C → S       C → S
  Duration          Ongoing    Instant   

In [9]:
# ==================================================
# EXERCISE 2.2: ACTION CLIENT
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 2.2: Action Client — Navigate Turtle to Goal")
print("=" * 80)

"""
📖 THEORY: Action Client

We'll write an action client that uses TurtleSim's built-in
RotateAbsolute action to rotate the turtle to specific headings
with feedback showing remaining rotation.
"""

rotate_client_py = '''
# ============================================================
# rotate_client.py — Action Client
# Uses TurtleSim's built-in RotateAbsolute action
#
# Save to: ~/ros2_ws/src/rotate_client.py
# Run with: python3 rotate_client.py
# ============================================================

import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from turtlesim.action import RotateAbsolute
import math


class RotateClient(Node):
    """
    Action client that rotates the turtle to specific headings.
    Demonstrates goal sending, feedback handling, and result.
    """

    def __init__(self):
        super().__init__("rotate_client")

        # Create action client
        # Arguments: node, action_type, action_name
        self._action_client = ActionClient(
            self,
            RotateAbsolute,
            "/turtle1/rotate_absolute",
        )

        self.get_logger().info("RotateClient ready")

    def rotate_to(self, theta_radians: float):
        """
        Send a goal to rotate turtle to the given heading.
        theta: target angle in radians (0=right, π/2=up, π=left)
        """
        # Wait for action server
        self.get_logger().info("Waiting for action server...")
        self._action_client.wait_for_server()

        # Build the goal
        goal = RotateAbsolute.Goal()
        goal.theta = theta_radians

        theta_deg = math.degrees(theta_radians)
        self.get_logger().info(
            f"Sending goal: rotate to {theta_deg:.1f}° ({theta_radians:.2f} rad)"
        )

        # Send goal with feedback callback
        send_goal_future = self._action_client.send_goal_async(
            goal,
            feedback_callback=self.feedback_callback,
        )

        # Register callback for when goal is accepted/rejected
        send_goal_future.add_done_callback(self.goal_response_callback)

    def goal_response_callback(self, future):
        """Called when the action server accepts or rejects the goal."""
        goal_handle = future.result()

        if not goal_handle.accepted:
            self.get_logger().error("Goal REJECTED by action server")
            return

        self.get_logger().info("Goal ACCEPTED — rotating...")

        # Register callback for when the action completes
        result_future = goal_handle.get_result_async()
        result_future.add_done_callback(self.result_callback)

    def feedback_callback(self, feedback_msg):
        """Called periodically while the action is executing."""
        remaining = feedback_msg.feedback.remaining
        remaining_deg = math.degrees(remaining)
        self.get_logger().info(
            f"  Feedback: {remaining_deg:.1f}° remaining"
        )

    def result_callback(self, future):
        """Called when the action completes."""
        result = future.result().result
        status = future.result().status

        delta_deg = math.degrees(result.delta)
        self.get_logger().info(
            f"✅ Action complete! Rotated {delta_deg:.1f}° | Status: {status}"
        )


def main(args=None):
    rclpy.init(args=args)
    node = RotateClient()

    # Rotate through 4 headings to draw a square-ish pattern
    headings = [0.0, math.pi/2, math.pi, -math.pi/2]   # 0°, 90°, 180°, 270°

    for theta in headings:
        node.rotate_to(theta)
        # Spin briefly to process callbacks
        rclpy.spin_once(node, timeout_sec=3.0)

    node.get_logger().info("All rotations complete!")
    node.destroy_node()
    rclpy.shutdown()


if __name__ == "__main__":
    main()
'''

print("📄 rotate_client.py:")
print(rotate_client_py)

print("""
📋 HOW TO RUN:
─────────────────────────────────────────────────────────────
  Terminal 1: ros2 run turtlesim turtlesim_node
  Terminal 2: python3 rotate_client.py

  Watch the turtle rotate to each heading with feedback
  showing remaining degrees.

  Or use CLI directly:
  ros2 action send_goal /turtle1/rotate_absolute \\
    turtlesim/action/RotateAbsolute "{theta: 1.5708}" --feedback

📋 KEY PARTS EXPLAINED:

  ActionClient(self, RotateAbsolute, "/turtle1/rotate_absolute")
    → Creates client for the RotateAbsolute action

  send_goal_async(goal, feedback_callback=self.feedback_callback)
    → Sends goal asynchronously
    → feedback_callback fires for each feedback message

  goal_response_callback → called when goal accepted/rejected
  feedback_callback      → called periodically during execution
  result_callback        → called when action completes

  This callback chain is the standard action client pattern in ROS2.
""")

print("\n✅ Exercise 2.2 Complete!")
print("=" * 80)


EXERCISE 2.2: Action Client — Navigate Turtle to Goal
📄 rotate_client.py:

# ============================================================
# rotate_client.py — Action Client
# Uses TurtleSim's built-in RotateAbsolute action
#
# Save to: ~/ros2_ws/src/rotate_client.py
# Run with: python3 rotate_client.py
# ============================================================

import rclpy
from rclpy.node import Node
from rclpy.action import ActionClient
from turtlesim.action import RotateAbsolute
import math


class RotateClient(Node):
    """
    Action client that rotates the turtle to specific headings.
    Demonstrates goal sending, feedback handling, and result.
    """

    def __init__(self):
        super().__init__("rotate_client")

        # Create action client
        # Arguments: node, action_type, action_name
        self._action_client = ActionClient(
            self,
            RotateAbsolute,
            "/turtle1/rotate_absolute",
        )

        self.get_logger().info("Ro

In [10]:
print("\n" + "=" * 80)
print("📊 PART 3: COMPARISON & PRACTICE")
print("=" * 80)


📊 PART 3: COMPARISON & PRACTICE


In [11]:
# ==================================================
# EXERCISE 3.1: COMPLETE COMPARISON
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.1: Topics vs Services vs Actions — Complete Comparison")
print("=" * 80)

print("""
📊 COMPLETE DECISION GUIDE:

  ┌─────────────────────────────────────────────────────────────┐
  │           WHEN TO USE WHAT IN ROS2                          │
  ├─────────────┬───────────────┬──────────────┬───────────────┤
  │             │   TOPIC       │   SERVICE    │   ACTION      │
  ├─────────────┼───────────────┼──────────────┼───────────────┤
  │ Pattern     │ Pub/Sub       │ Req/Res      │ Goal/FB/Result│
  │ Duration    │ Continuous    │ Instant      │ Long-running  │
  │ Feedback    │ N/A           │ N/A          │ Yes           │
  │ Cancellable │ N/A           │ No           │ Yes           │
  │ Many-to-Many│ Yes           │ No           │ No            │
  │ Async       │ Yes           │ Blocking     │ Yes           │
  └─────────────┴───────────────┴──────────────┴───────────────┘

📋 REAL-WORLD EXAMPLES:

  Situation                              Use
  ────────────────────────────────────────────────────────────
  Camera publishing 30fps images         Topic
  LiDAR publishing 10Hz scan data        Topic
  Turtle position streaming              Topic
  Velocity commands to motors            Topic

  Spawn a new turtle                     Service
  Reset the simulation                   Service
  Get robot battery level                Service
  Set LED color                          Service
  Change a parameter                     Service

  Navigate to a waypoint (30 seconds)   Action
  Pick up an object                      Action
  Scan a room with LiDAR                Action
  Execute a complex motion sequence      Action
  Rotate turtle to specific heading      Action ← (TurtleSim example!)

📋 SECURITY SYSTEM (WEEK 14) EQUIVALENTS:

  ROS2 Concept         Week 14 FastAPI Equivalent
  ─────────────────────────────────────────────────────────
  Topic (pub/sub)      Redis pub/sub or streaming endpoint
  Service              POST /api/v1/detections (quick)
  Action               Background task + polling endpoint
  Node                 FastAPI service / Docker container
  Package              Docker image / Python package
  Launch file          docker-compose.yml
""")

print("\n✅ Exercise 3.1 Complete!")
print("=" * 80)


EXERCISE 3.1: Topics vs Services vs Actions — Complete Comparison

📊 COMPLETE DECISION GUIDE:

  ┌─────────────────────────────────────────────────────────────┐
  │           WHEN TO USE WHAT IN ROS2                          │
  ├─────────────┬───────────────┬──────────────┬───────────────┤
  │             │   TOPIC       │   SERVICE    │   ACTION      │
  ├─────────────┼───────────────┼──────────────┼───────────────┤
  │ Pattern     │ Pub/Sub       │ Req/Res      │ Goal/FB/Result│
  │ Duration    │ Continuous    │ Instant      │ Long-running  │
  │ Feedback    │ N/A           │ N/A          │ Yes           │
  │ Cancellable │ N/A           │ No           │ Yes           │
  │ Many-to-Many│ Yes           │ No           │ No            │
  │ Async       │ Yes           │ Blocking     │ Yes           │
  └─────────────┴───────────────┴──────────────┴───────────────┘

📋 REAL-WORLD EXAMPLES:

  Situation                              Use
  ──────────────────────────────────────────────────

In [12]:
# ==================================================
# EXERCISE 3.2: PRACTICE CHALLENGES
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.2: Practice Challenges")
print("=" * 80)

print("""
🎯 PRACTICE CHALLENGES — Complete in WSL2:

  ─── Challenge 1: Use Built-in TurtleSim Services ────────────
  □ Reset the canvas:
    ros2 service call /turtlesim/reset std_srvs/srv/Empty
  □ Teleport to corner (1, 1):
    ros2 service call /turtle1/teleport_absolute \\
      turtlesim/srv/TeleportAbsolute "{x: 1.0, y: 1.0, theta: 0.0}"
  □ Change pen to green, width 5:
    ros2 service call /turtle1/set_pen \\
      turtlesim/srv/SetPen "{r: 0, g: 255, b: 0, width: 5, off: 0}"
  □ Move the turtle and admire the green trail

  ─── Challenge 2: Run the Commander Server ───────────────────
  □ Terminal 1: ros2 run turtlesim turtlesim_node
  □ Terminal 2: python3 turtle_commander_server.py
  □ Terminal 3: python3 turtle_commander_client.py go_home
  □ Terminal 3: python3 turtle_commander_client.py spin
  □ Terminal 3: python3 turtle_commander_client.py stop
  □ Check rqt_graph — see the service connections

  ─── Challenge 3: RotateAbsolute Action ──────────────────────
  □ Terminal 1: ros2 run turtlesim turtlesim_node
  □ Terminal 2: python3 rotate_client.py
  □ Watch turtle rotate to each heading with feedback
  □ Try modifying headings list to rotate to 45°, 135°, 225°, 315°
    (math.pi/4, 3*math.pi/4, -3*math.pi/4, -math.pi/4)

  ─── Challenge 4: Cancel an Action (CLI) ─────────────────────
  □ Send a long rotation goal in Terminal 2:
    ros2 action send_goal /turtle1/rotate_absolute \\
      turtlesim/action/RotateAbsolute "{theta: 3.14159}" --feedback
  □ While it's rotating, press Ctrl+C to cancel
  □ Observe: turtle stops mid-rotation
  □ This is what action cancellation looks like!

  ─── Challenge 5: Inspect Everything ─────────────────────────
  □ ros2 service list     → how many services are available?
  □ ros2 action list      → how many actions are available?
  □ ros2 action info /turtle1/rotate_absolute
  □ ros2 interface show turtlesim/action/RotateAbsolute
""")

print("\n✅ Exercise 3.2 Complete!")
print("=" * 80)


EXERCISE 3.2: Practice Challenges

🎯 PRACTICE CHALLENGES — Complete in WSL2:

  ─── Challenge 1: Use Built-in TurtleSim Services ────────────
  □ Reset the canvas:
    ros2 service call /turtlesim/reset std_srvs/srv/Empty
  □ Teleport to corner (1, 1):
    ros2 service call /turtle1/teleport_absolute \
      turtlesim/srv/TeleportAbsolute "{x: 1.0, y: 1.0, theta: 0.0}"
  □ Change pen to green, width 5:
    ros2 service call /turtle1/set_pen \
      turtlesim/srv/SetPen "{r: 0, g: 255, b: 0, width: 5, off: 0}"
  □ Move the turtle and admire the green trail

  ─── Challenge 2: Run the Commander Server ───────────────────
  □ Terminal 1: ros2 run turtlesim turtlesim_node
  □ Terminal 2: python3 turtle_commander_server.py
  □ Terminal 3: python3 turtle_commander_client.py go_home
  □ Terminal 3: python3 turtle_commander_client.py spin
  □ Terminal 3: python3 turtle_commander_client.py stop
  □ Check rqt_graph — see the service connections

  ─── Challenge 3: RotateAbsolute Action ────────

In [13]:
# ==================================================
# EXERCISE 3.3: DAY 101 SUMMARY
# ==================================================

print("\n" + "=" * 80)
print("EXERCISE 3.3: Day 101 Summary")
print("=" * 80)

print("""
📚 WHAT WE LEARNED TODAY:

✅ ROS2 Services:
   • Request/Response — synchronous, one-to-one
   • Use for quick commands where you need a result back
   • Service server: create_service(type, name, callback)
   • Service client: create_client(type, name) + call_async()
   • Always wait_for_service() before calling
   • Trigger service: empty request, bool+string response

✅ ROS2 Actions:
   • Goal/Feedback/Result — async, long-running, cancellable
   • Use for tasks > 1 second or when progress updates needed
   • Action client: ActionClient + send_goal_async()
   • Three callbacks: goal_response, feedback, result
   • TurtleSim's RotateAbsolute is a perfect learning example

✅ Complete Communication Comparison:
   • Topic  → streaming sensor data, continuous commands
   • Service → quick commands, queries, configuration
   • Action → navigation, manipulation, long sequences

💡 KEY INSIGHTS:
   1. Services block the client — don't use for long tasks
      A 10-second service call freezes the client node entirely.
      That's what Actions are for.

   2. The callback chain in actions is deliberate
      goal_response → feedback → result
      Each step is asynchronous — the node stays responsive
      to other callbacks while waiting for the action.

   3. Most real robot software uses all three
      Camera data → Topic (62 Hz)
      Spawn robot → Service (instant)
      Navigate to goal → Action (30+ seconds with feedback)
""")

print("\n✅ Exercise 3.3 Complete!")
print("=" * 80)


EXERCISE 3.3: Day 101 Summary

📚 WHAT WE LEARNED TODAY:

✅ ROS2 Services:
   • Request/Response — synchronous, one-to-one
   • Use for quick commands where you need a result back
   • Service server: create_service(type, name, callback)
   • Service client: create_client(type, name) + call_async()
   • Always wait_for_service() before calling
   • Trigger service: empty request, bool+string response

✅ ROS2 Actions:
   • Goal/Feedback/Result — async, long-running, cancellable
   • Use for tasks > 1 second or when progress updates needed
   • Action client: ActionClient + send_goal_async()
   • Three callbacks: goal_response, feedback, result
   • TurtleSim's RotateAbsolute is a perfect learning example

✅ Complete Communication Comparison:
   • Topic  → streaming sensor data, continuous commands
   • Service → quick commands, queries, configuration
   • Action → navigation, manipulation, long sequences

💡 KEY INSIGHTS:
   1. Services block the client — don't use for long tasks
      A

In [14]:
print("\n" + "=" * 80)
print("🎯 DAY 101 COMPLETE! ✅")
print("=" * 80)

print("""
OBJECTIVES ACHIEVED:
   ✅ Understood service request/response pattern
   ✅ Written turtle_commander_server.py with 3 service endpoints
   ✅ Written turtle_commander_client.py with async service calls
   ✅ Understood action goal/feedback/result pattern
   ✅ Written rotate_client.py using TurtleSim's RotateAbsolute action
   ✅ Mastered Topics vs Services vs Actions decision framework

💡 KEY FILES CREATED TODAY:
   ~/ros2_ws/src/
   ├── turtle_commander_server.py  ← Service server (go_home, spin, stop)
   ├── turtle_commander_client.py  ← Service client (calls commander)
   └── rotate_client.py            ← Action client (RotateAbsolute)

🎯 TOMORROW (DAY 102):
   - ROS2 Packages — proper project structure
   - Create a proper ROS2 package with colcon build
   - Launch files — start multiple nodes with one command
   - Parameters — configurable nodes without code changes
   - Build the foundation for the Week 15 TurtleSim project

Week 15 Progress: 42.9% (3/7 days complete)
Overall Progress: Day 101/168 (60.1% complete)
""")

print("=" * 80)


🎯 DAY 101 COMPLETE! ✅

OBJECTIVES ACHIEVED:
   ✅ Understood service request/response pattern
   ✅ Written turtle_commander_server.py with 3 service endpoints
   ✅ Written turtle_commander_client.py with async service calls
   ✅ Understood action goal/feedback/result pattern
   ✅ Written rotate_client.py using TurtleSim's RotateAbsolute action
   ✅ Mastered Topics vs Services vs Actions decision framework

💡 KEY FILES CREATED TODAY:
   ~/ros2_ws/src/
   ├── turtle_commander_server.py  ← Service server (go_home, spin, stop)
   ├── turtle_commander_client.py  ← Service client (calls commander)
   └── rotate_client.py            ← Action client (RotateAbsolute)

🎯 TOMORROW (DAY 102):
   - ROS2 Packages — proper project structure
   - Create a proper ROS2 package with colcon build
   - Launch files — start multiple nodes with one command
   - Parameters — configurable nodes without code changes
   - Build the foundation for the Week 15 TurtleSim project

Week 15 Progress: 42.9% (3/7 days c